# Lineshape-parameter diagnostics — E791 Fit 2

This notebook isolates the failure mode seen when resonance mass/width parameters are released. It compares the dominant $\sigma\pi^+$ contribution with the weak $\rho(1450)\pi^+$ component.

Every `DecayModel` normalizes its dynamical components to unit phase-space integral by default and owns its normalization MC internally. Standard fits below therefore call `model.prepare_cache(data)` without passing an external normalization sample.

The final section deliberately changes `normalization_size` from 100k to 1M as a controlled numerical-stability test.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()


## 1. Common Fit-2 inputs


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma": (1.17, 205.7), "rho770": (1.0, 0.0),
    "NR": (0.48, 57.3), "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3), "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r*np.cos(phase), r*np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}
resonance_data = {
    "sigma": (0.478, 0.324, 0),
    "rho770": (0.7693, 0.1502, 1),
    "f0_980": (0.975, 0.044, 0),
    "f2_1270": (1.275, 0.185, 2),
    "f0_1370": (1.434, 0.173, 0),
    "rho1450": (1.465, 0.310, 1),
}


## 2. Controlled model builder


In [ ]:
def build_model(
    dynamic=None,
    free_component_coefficients=(),
    free_all_coefficients=False,
    normalization_size=1_000_000,
):
    truth = {}
    coefficients = {}
    for name in truth_xy:
        x0, y0 = truth_xy[name]
        if name == "rho770":
            coefficients[name] = RealImag(1.0, 0.0)
            continue
        free = free_all_coefficients or name in free_component_coefficients
        if free:
            x = Parameter.coefficient(f"{name}.x", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01)
            y = Parameter.coefficient(f"{name}.y", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01)
            coefficients[name] = RealImag(x, y)
            truth[x.name], truth[y.name] = x0, y0
        else:
            coefficients[name] = RealImag(x0, y0)

    components = []
    for name in ("sigma", "rho770", "f0_980", "f2_1270", "f0_1370", "rho1450"):
        mass0, width0, spin = resonance_data[name]
        mass, width = mass0, width0
        if name == dynamic:
            mass = Parameter.dynamics(
                f"{name}.mass", mass0*0.95, owner=name,
                bounds=(max(0.20, mass0*0.70), mass0*1.30), step=0.002,
            )
            width = Parameter.dynamics(
                f"{name}.width", width0*1.15, owner=name,
                bounds=(max(0.01, width0*0.35), width0*2.0), step=0.003,
            )
            truth[mass.name], truth[width.name] = mass0, width0
        components.append(
            Resonance(
                name, (0,1), coefficients[name], mass=mass, width=width, spin=spin,
                resonance_radius=3.0, parent_radius=3.0,
            )
        )
    components.append(NonResonant(coefficients["NR"]))
    model = DecayModel(
        channel, components,
        normalization_size=normalization_size,
        normalization_seed=2027,
    )
    return model, truth

truth_model, _ = build_model()
print("default normalization events:", truth_model.normalization_size)


## 3. One common pseudo-data sample

The pseudo-data candidate pool is independent of the normalization MC. The truth density is evaluated with the same normalized-component convention used in all fits.


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000

pool = truth_model.generate_phase_space(N_POOL, seed=2000)
truth_weight = pool.weights * truth_model.intensity(pool.as_dict())
data = weighted_resample(jax.random.key(791), pool, truth_weight, N_DATA, replace=True)
print("data", data.size, "truth normalization MC", truth_model.normalization_sample.size)


## 4. Fit helper


In [ ]:
def run_fit(label, model, truth, n_starts=12):
    cache = model.prepare_cache(data)
    def nll(values):
        intensity, normalization = cache.evaluate(values)
        return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

    minimizer = Minimizer(nll, model.parameters)
    scan = minimizer.fit_multistart(
        n_starts=n_starts, seed=314159, include_default=False, simplex=True
    )
    result = scan.best
    print("\n" + label)
    print("normalization events =", model.normalization_sample.size)
    print("valid =", result.valid, "NLL =", float(result.fval), "EDM =", float(result.fmin.edm))
    print("NLL(best)-NLL(truth) =", float(result.fval - nll(truth)))
    print(f"{'parameter':16s} {'truth':>11s} {'fit':>11s} {'error':>11s} {'pull':>10s}")
    for p in model.parameters:
        if p.fixed:
            continue
        fit = float(result.values[p.name])
        err = float(result.errors[p.name])
        pull = (truth[p.name]-fit)/err
        print(f"{p.name:16s} {truth[p.name]:11.6f} {fit:11.6f} {err:11.6f} {pull:10.3f}")
    return cache, nll, scan, result


## 5. Dominant $\sigma$: mass and width only


In [ ]:
model_sigma_shape, truth_sigma_shape = build_model(dynamic="sigma")
res_sigma_shape = run_fit("sigma mass/width only", model_sigma_shape, truth_sigma_shape)


## 6. $\sigma$: coefficient + mass + width


In [ ]:
model_sigma4, truth_sigma4 = build_model(dynamic="sigma", free_component_coefficients=("sigma",))
res_sigma4 = run_fit("sigma x/y/mass/width", model_sigma4, truth_sigma4)


## 7. Full coefficient fit + $m_\sigma,\Gamma_\sigma$


In [ ]:
model_sigma_full, truth_sigma_full = build_model(dynamic="sigma", free_all_coefficients=True)
res_sigma_full = run_fit("all coefficients + sigma mass/width", model_sigma_full, truth_sigma_full, n_starts=20)


## 8. Weak $\rho(1450)$: coefficient + mass + width


In [ ]:
model_rho4, truth_rho4 = build_model(dynamic="rho1450", free_component_coefficients=("rho1450",))
res_rho4 = run_fit("rho1450 x/y/mass/width", model_rho4, truth_rho4)


## 9. Full coefficient fit + $m_{\rho(1450)},\Gamma_{\rho(1450)}$


In [ ]:
model_rho_full, truth_rho_full = build_model(dynamic="rho1450", free_all_coefficients=True)
res_rho_full = run_fit("all coefficients + rho1450 mass/width", model_rho_full, truth_rho_full, n_starts=20)


## 10. Internal normalization-MC sensitivity: 100k versus 1M

This intentionally constructs two otherwise identical models with different internal normalization statistics. Because component normalization itself is estimated from MC, the fitted complex coefficients may move slightly; the key diagnostic is the stability of the shape parameters and NLL profile.


In [ ]:
model_rho_100k, truth_rho_100k = build_model(
    dynamic="rho1450", free_component_coefficients=("rho1450",), normalization_size=100_000
)
_, _, _, rho_100k = run_fit("rho1450 x/y/mass/width — 100k internal normalization", model_rho_100k, truth_rho_100k)
_, _, _, rho_1m = res_rho4

for name in ("rho1450.mass", "rho1450.width"):
    small = float(rho_100k.values[name])
    large = float(rho_1m.values[name])
    print(f"{name:16s}: 100k={small:.6f}  1M={large:.6f}  shift={small-large:+.6f}")


## 11. Local NLL curvature at truth


In [ ]:
def scan_coordinate(model, truth, name, grid):
    cache = model.prepare_cache(data)
    def nll(values):
        intensity, normalization = cache.evaluate(values)
        return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)
    values = []
    for x in grid:
        point = dict(truth); point[name] = float(x)
        values.append(float(nll(point)))
    values = np.asarray(values)
    return values-values.min()

sigma_m_grid = np.linspace(0.40, 0.56, 60)
rho_m_grid = np.linspace(1.30, 1.60, 60)
sigma_dnll = scan_coordinate(model_sigma4, truth_sigma4, "sigma.mass", sigma_m_grid)
rho_dnll = scan_coordinate(model_rho4, truth_rho4, "rho1450.mass", rho_m_grid)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(sigma_m_grid, sigma_dnll, label=r"$m_\sigma$")
ax.plot(rho_m_grid, rho_dnll, label=r"$m_{\rho(1450)}$")
ax.axhline(0.5, linestyle="--", linewidth=1)
ax.set(xlabel="mass [GeV]", ylabel=r"$\Delta$NLL (others fixed)", ylim=(0,10))
ax.legend(); plt.show()
